# 머신러닝 기초

> **노트북 사용 전 확인할 것**

> 셀을 순서대로 실행하지 않으면 변수 상태가 꼬일 수 있다.\
> 위에서 아래 순서대로 실행하면서 확인할 것.

> 활성화 하지 않은 상태로 M 클릭 시 Markdown\
> 활성화 하지 않은 상태로 Y 클릭 시 Python 코드

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from _style import setup
from _ml import build_dataset, FEATURES, SEED

setup()
pd.set_option("display.width", 130)
pd.set_option("display.max_column", 20)
print("준비 완료")


### 1. 프로그래밍(코딩)과의 차이점
지금까지 우리가 작성해온 프로그래밍은 **규칙을 사람이 정하는 것**이었다.
```python
if volume > 100_000 and changeRate > 0.05:
    return "급등주"
```
#### 규칙이 명확할 때, (설명이 가능한 상황)
=> 프로그래밍이 빠르고, 정확하고 좋을 수 있다.
#### 규칙을 사람이 알 수 없을 때, (설명이 어려워지는 상황)
> "오늘 주가가 오를 종목의 조건은 무엇인가?"
>
거래량, 등락률, 이동평균, ... 등 수십 개 이상의 변수로 조건을 작성할 수 있겠지만,\
**머신러닝은 데이터에서 그 규칙을 컴퓨터가 찾아내게 하는 방법**이게 때문에 이 문제를 해소할 수 있다.

In [ ]:
df = build_dataset()

# 규칙: 거래량이 평소보다 2배 이상 많고,
#       20일 평균보다 3% 이상 높으면 오를 것이다
rule = (df["volume_ratio"] >= 2.0) & (df["ma20_ratio"] >= 1.03)

# df["target_up"] -> 올랐으면 1, 아니면 0
hit = (rule == df["target_up"]).mean()

print(f"규칙에 해당하는 날: {rule.sum():,}건 / {len(df):,}건")
print(f"규칙의 적중률: {hit:.4f}")
print(f"전체 상승 비율: {df['target_up'].mean()}")

> **적중률이 생각보다 낮다**\
>
> 사람이 느낌으로 만든 규칙이므로 어떤 조건을 어떤 값으로 잡아야 할지\
구분하는 것에 어려움이 있음. 조건을 바꿔가며 직접 찾을 수도 있긴 하나,\
만약 피처(변수)가 7개만 되더라도 조합이 폭발적으로 많아짐.

- 프로그래밍: 데이터 + 규칙(사람이 작성)을 입력해서 결과를 받는다.
- 머신러닝: 데이터 + 정답을 입력해서 규칙을 만든다.

> 규칙이 명확하면 직접 프로그래밍으로 답을 도출하고,\
> **규칙을 알 수 없을 때** 머신러닝을 이용하여 규칙을 만든다.

---
### 분류와 회귀
**예측할 대상**
||분류|회귀|
|:--:|:--:|:--:|
|에측 가능한 것|범주|수치|
|질문|오를까, 내릴까?|얼마나 오를까?
|출력|up/down|2.7%|
|평가지표|정확도, 정밀도, 재현율|MAE, RMSE, ...|

**동일한 데이터로 두 문제의 답을 모두 도출할 수 있음!**


In [ ]:
# target_ret: 수익률 - 수치 => 회귀
# target_up: 올랐는지, 오르지 않았는지 (1/0) - 범주 => 분류

sample = df[["code", "date", "target_ret", "target_up"]].head(8)
print(sample.to_string(index=False))

print("같은 행이지만 정답을 2가지로 확인 가능 (회귀와 분류)")

> **평가 지표는 섞어서 사용 불가**
>
> 분류 모델에 MAE, RMSE, ...을 사용하거나\
> 회귀 모델에 정확도를 사용할 수 있을까?\
> => 사용할 수 없음.\
> "정확도 92%"라는 판단은, **분류에서만** 의미가 있고,\
> 회귀에서는 "정확하게 맞췄다"라는 개념 자체가 성립하지 않는다.

### 기준선 -> 사람이 정해야 함!
연속된 값을 범주로 변경하려면 어디서 자를 것인지 정해야 함\
-> 정답의 기준이 됨

In [ ]:
# 기준선을 바꿔가면서 "상승"의 비율이 어떻게 변하는지 확인

base_lines = [0.0, 0.01, 0.02, 0.03, 0.05]
print(f"{'기준선':>10}{'양성비율':>12}{'양성건수':>12}")
print("-" * 60)

for th in base_lines:
    positive = (df["target_ret"] > th)
    print(f"{th:>9.0%}{positive.mean():>15.2%}{positive.sum():>16,}")

> **기준선이 결과를 바꾼다**
>
> 0%를 기준으로 하면 상승, 하락이 거의 반반이다.\
> 5%를 기준으로 하면 상승이 약 1.4%밖에 안 된다.\
> 이런 불균형 데이터에서 "전부 하락"이라고 하면 정확도가 98% 이상이다.

---
### 학습, 검증 데이터 분리
시험 문제를 미리 알려주고 시험을 보면 당연히 높은 점수가 나올 것이다.\
모델도 동일하다. **학습에 사용한 데이터로 평가하면 성능이 과대평가**된다.\

|데이터|용도|비율|
|:--:|:--:|:--:|
|학습|모델이 규칙을 학습(배움)|60~70%|
|검증|하이퍼 파라미터|15~20%|
|테스트|최종 성능 측정|15~20%|

In [ ]:
from sklearn.model_selection import train_test_split

small = df.sample(5_000, random_state=SEED)
# small.head()

# 입력 데이터 (피처, 독립변수)
X = small[FEATURES]
# 정답 데이터 (레이블, 종속변수)
y = small["target_up"]

X_train, X_test, y_train, y_test = train_test_split(
    # 입력, 정답
    X, y,
    # 학습 : 테스트 = 8 : 2
    test_size=0.2,
    # 재현성을 위해 SEED를 정함
    random_state=SEED,
    # 타겟레벨(y) 클래스 비율을 그대로 유지하며 학습 데이터와 테스트 데이터 분리
    stratify=y,
)

print(f"X shape: {X.shape} (행수, 피처수)")
print(f"y shape: {y.shape} (1차원)")
print()

print(f"학습 데이터: {len(X_train):,}행 / 상승 비율: {y_train.mean():.4f}")
print(f"테스트: {len(X_test):,}행 / 상승 비율: {y_test.mean():.4f}")

---
### 시계열 데이터는 랜덤 분할하면 안됨!
시세 데이터는 날짜가 있는 시계열 데이터이고,\
이전과 동일하게 train_test_split 함수를 사용하면 문제가 될 수 있음.

In [ ]:
# 랜덤 분할
# (train_test_split 함수는 기본적으로 랜덤 분할)
rnd_train, rnd_test = train_test_split(df, test_size=0.2, 
                                       random_state=SEED)

# 시점 분할
# -> 직접 날짜를 기준으로 하여 앞뒤로 자른다.
# quantile(0.8): 상위 80% 지점의 날짜
cutoff = df["date"].quantile(0.8)

seq_train = df[df["date"] <= cutoff]
seq_test = df[df["date"] > cutoff]

print("=== 랜덤 분할 ===")
print(f"{rnd_train.date.min().date()} ~ {rnd_train.date.max().date()}")
print(f"{rnd_test.date.min().date()} ~ {rnd_test.date.max().date()}")
print()
print("=== 시점 분할 ===")
print(f"{seq_train.date.min().date()} ~ {seq_train.date.max().date()}")
print(f"{seq_test.date.min().date()} ~ {seq_test.date.max().date()}")

---
### scikit-learn 공통 인터페이스
**모든 모델이 같은 방식으로 동작하는 것**이 scikit-learn의 큰 장점

```python
model = 모델생성()

model.fit(X_train, y_train)             # 학습
pred = model.predict(X_test)            # 예측
score = model.score(X_test, y_test)     # 평가
```

In [ ]:
# 결정 트리 분류 모델
from sklearn.tree import DecisionTreeClassifier
# 결정 트리 랜덤 포레스트
from sklearn.ensemble import RandomForestClassifier
# 로지스틱 회귀는 이름만 회귀이고 사실 분류 모델
from sklearn.linear_model import LogisticRegression

# 시점 분할한 데이터 사용
# 시점 분할을 완료한 데이터이므로 랜덤 분할을 해줘도 괜찮음
train_s = seq_train.sample(8_000, random_state=SEED)
test_s = seq_test.sample(2_000, random_state=SEED)

Xtr, ytr = train_s[FEATURES], train_s["target_up"]
Xte, yte = test_s[FEATURES], test_s["target_up"]

# 모델 생성
models = {
    "LogisticRegression": LogisticRegression(max_iter=1_000, random_state=SEED),
    "DecisionTress": DecisionTreeClassifier(max_depth=5, random_state=SEED),
    "RandomForest": RandomForestClassifier(max_depth=5, random_state=SEED,
                                           n_estimators=50, n_jobs=-1)
}

for name, model in models.items():
    model.fit(Xtr, ytr)             # 학습 (입력, 정답)
    pred = model.predict(Xte)       # 예측 (입력)
    score = model.score(Xte, yte)   # 평가 (입력, 정답) => 검증용/테스트용

    print(f"{name:<20} score {score:.4f}")

|메소드|역할|비고|
|:--:|:--:|:--:|
|fit(X, y)|학습||
|predict(X)|예측|검증용, 테스트용 데이터만 입력해야 함.|
|predict_proba(X)|확률(분류)||
|score(X, y)|기본평가지표||

> score() 메서드는 모델마다 의미가 달라진다.
> 분류 모델에서는 정확도를 의미하고,\
> 회귀 모델이면 어떤 측정 수치를 반환함.

In [ ]:
# predict와 predict_proba 비교

rf = models["RandomForest"]

pred = rf.predict(Xte)          # 0 또는 1
proba = rf.predict_proba(Xte)   # 각 클래스에 속할 확률

print(f"predict:        shape {pred.shape} 상위 5개 {pred[:5]}")
print(f"prdeict_proba:  shape{proba.shape}")
print(f"predict_proba 상위 5개")
print(pd.DataFrame(proba[:5], columns=["하락또는유지확률", "상승확률"]).round(4))

---
- 머신러닝: 규칙을 알 수 없을 때 데이터에서 규칙을 찾아내는 방법
- 분류 vs 회귀: 범위(범주)에 해당하는지 여부 vs 수치에 해당하는지 여부
- 데이터 분할: 학습용 데이터와 검증용 데이터로 나누어 사용.\
                검증용 데이터는 마지막에 한 번만 사용.
    - 시계열 데이터: 랜덤 분할 금지! **시점을 기준으로 구분**할 것.
- 과적합: 학습과 테스트 결과의 차이가 심할 때\
            (모델이 너무 정답에만 의존한다는 신호)